In [16]:


import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
import plotly.express as px
import plotly.figure_factory as ff

In [17]:
PROCESSED_DIR = "../processed"
file_path = os.path.join(PROCESSED_DIR, "merged_daily.csv")
df = pd.read_csv(file_path)
df['date'] = pd.to_datetime(df['date'])
df.head()

FIGURES_DIR = "../figures"


In [18]:
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(
    go.Bar(
        x=df['date'],
        y=df['prcp'],
        name="Rainfall (mm)",
        marker_color='rgba(0, 100, 255, 0.3)',
        hovertemplate='%{y} mm<extra></extra>'
    ),
    secondary_y=True, 
)
fig.add_trace(
    go.Scatter(
        x=df['date'],
        y=df['tavg'],
        name="Average temperature (°C)",
        mode='lines',
        line=dict(color='orange', width=2, dash='dot'),
        hovertemplate='%{y} °C<extra></extra>'
    ),
    secondary_y=True, 
)
fig.add_trace(
    go.Scatter(
        x=df['date'],
        y=df['pm2_5'],
        name="PM2.5 (µg/m³)",
        mode='lines',
        line=dict(color='red', width=3),
        hovertemplate='%{y} µg/m³<extra></extra>'
    ),
    secondary_y=False, 
)

fig.update_layout(
    title=dict(
        text='<b>Relationship: PM2.5 - temperature - rainfall</b>',
        x=0.5,
        font=dict(size=20)
    ),
    xaxis=dict(
        title="Date",
        showgrid=False
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    hovermode="x unified",
    template="plotly_white"
)

fig.update_yaxes(
    title_text="<b>PM2.5 (µg/m³)</b>", 
    title_font=dict(color="red"),
    tickfont=dict(color="red"),
    secondary_y=False
)

fig.update_yaxes(
    title_text="<b>Temperature (°C) / Rainfall (mm)</b>", 
    title_font=dict(color="navy"),
    tickfont=dict(color="navy"),
    secondary_y=True,
    showgrid=False
)

In [19]:
fig.write_image("../figures/pm25-temp-rain.pdf")

In [20]:
def categorize_rain(x):
    if x == 0: return 'No rain'
    elif x < 10: return 'Light rain (<10mm)'
    else: return 'Moderate / heavy rain (≥10mm)'

df['Rain_Category'] = df['prcp'].apply(categorize_rain)
rain_summary = df.groupby('Rain_Category')['pm2_5'].mean().reindex(
    ['No rain', 'Light rain (<10mm)', 'Moderate / heavy rain (≥10mm)']
)

# 2. Wind categories
bins_wind = [0, 5, 10, 15, 100]
labels_wind = ['Calm (0–5)', 'Breeze (5–10)', 'Windy (10–15)', 'Strong wind (>15)']
df['Wind_Category'] = pd.cut(df['wspd'], bins=bins_wind, labels=labels_wind)
wind_summary = df.groupby('Wind_Category', observed=True)['pm2_5'].mean()

# 3. Temperature categories
bins_temp = [0, 20, 28, 100]
labels_temp = ['Cool (<20°C)', 'Mild/Warm (20–28°C)', 'Hot (>28°C)']
df['Temp_Category'] = pd.cut(df['tavg'], bins=bins_temp, labels=labels_temp)


# --- Layout ---
fig = make_subplots(
    rows=2, cols=2,
    specs=[[{"type": "bar"}, {"type": "scatter"}], 
           [{"colspan": 2, "type": "xy"}, None]],
    subplot_titles=(
        "1. More rain → lower PM2.5", 
        "2. Stronger wind → cleaner air", 
        "3. PM2.5 distribution by temperature (Violin plot)"
    ),
    vertical_spacing=0.15,
    horizontal_spacing=0.10
)

# 1) Rain bar chart
colors_rain = ['#d62728', '#ffbf00', '#2ca02c']  # red → yellow → green
fig.add_trace(
    go.Bar(
        x=rain_summary.index, 
        y=rain_summary.values,
        marker_color=colors_rain,
        text=rain_summary.values.round(1),
        textposition='auto',
        name="Rainfall"
    ),
    row=1, col=1
)

# 2) Wind area chart
fig.add_trace(
    go.Scatter(
        x=wind_summary.index,
        y=wind_summary.values,
        mode='lines+markers',
        line=dict(width=3, color='#2ca02c'),
        fill='tozeroy',
        fillcolor='rgba(44,160,44,0.25)',
        marker=dict(size=10, color='darkgreen'),
        name="Wind speed"
    ),
    row=1, col=2
)

# 3) Temperature vs PM2.5 (Violin plot)
colors_temp = ['#1f77b4', '#ff7f0e', '#d62728']  # cool → warm → hot

for i, cat in enumerate(labels_temp):
    subset = df[df['Temp_Category'] == cat]
    fig.add_trace(
        go.Violin(
            x=subset['Temp_Category'],
            y=subset['pm2_5'],
            name=cat,
            box_visible=True,
            meanline_visible=True,
            points=False,
            line_color=colors_temp[i],
            opacity=0.7,
            side='positive',
            width=0.8
        ),
        row=2, col=1
)

# WHO guideline band for PM2.5
fig.add_hrect(
    y0=0, y1=15, 
    line_width=0, 
    fillcolor="green", 
    opacity=0.1,
    annotation_text="WHO safe range",
    annotation_position="bottom right",
    row=2, col=1
)

# Layout
fig.update_layout(
    title="<b>Factors related to PM2.5</b>",
    title_font=dict(size=22),
    height=850,
    template="plotly_white",
    margin=dict(l=40, r=40, t=90, b=40),
    showlegend=False,
    violinmode='group'
)

# Axis labels
fig.update_yaxes(title_text="Average PM2.5 (µg/m³)", row=1, col=1)
fig.update_yaxes(title_text="Average PM2.5 (µg/m³)", row=1, col=2)
fig.update_yaxes(title_text="PM2.5 density", row=2, col=1)

In [21]:
fig.write_image("../figures/dashboard_rain_wind_temp.pdf")

In [22]:
cols_map = {
    # Air pollutants
    'pm2_5': 'PM2.5',
    'pm10': 'PM10',
    'carbon_monoxide': 'CO (traffic/combustion)',
    'nitrogen_dioxide': 'NO2 (traffic)',
    'sulphur_dioxide': 'SO2 (industry)',
    'ozone': 'O3 (ozone)',
    
    # Weather
    'tavg': 'Average temperature',
    'tmax': 'Max temperature',
    'prcp': 'Rainfall',
    'wspd': 'Wind speed',
    'pres': 'Pressure',
    'uv_index_max': 'UV index'
}

# Select available columns and rename
# Keep only numeric columns for correlation
available_cols = [c for c in cols_map.keys() if c in df.columns]
df_clean = df[available_cols].rename(columns=cols_map)

# Pearson correlation matrix
corr_matrix = df_clean.corr().round(2)

# Plot heatmap
fig = px.imshow(
    corr_matrix,
    text_auto=True,
    aspect="auto",
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    title="<b>Correlation matrix: air pollutants & weather</b>"
)

fig.update_layout(
    height=900, 
    width=1000,
    template="plotly_white",
    title_font_size=20,
    title_x=0.5
)

# Annotation explaining how to read the chart
fig.add_annotation(
    text="🟥 RED (+): move together (e.g. PM2.5 & PM10)<br>🟦 BLUE (−): opposite direction (e.g. stronger wind → lower PM2.5)",
    xref="paper", yref="paper",
    x=0.5, y=-0.15,
    showarrow=False,
    font=dict(size=14, color="black"),
    align="center",
    bgcolor="rgba(255,255,255,0.8)",
    bordercolor="black",
    borderwidth=1
)

# Rotate X labels for readability
fig.update_xaxes(side="bottom", tickangle=-45)

In [23]:
# Correlation between heavy-rain days (>20mm) and PM2.5
heavy_rain = df[df['prcp'] > 20]
if len(heavy_rain) == 0:
    print("No days with rainfall > 20 mm in the dataset.")
else:
    r_heavy = heavy_rain[['prcp', 'pm2_5']].corr().iloc[0, 1]
    print(f"Heavy-rain subset size: {len(heavy_rain)} days")
    print(f"Pearson r (prcp vs pm2_5) for prcp > 20mm: {r_heavy:.3f}")

Heavy-rain subset size: 22 days
Pearson r (prcp vs pm2_5) for prcp > 20mm: -0.290


In [24]:
fig.write_image("../figures/corr_heatmap.pdf")

In [25]:
# 1. Month column
df['Month'] = df['date'].dt.month

# 2. Season column
def define_season(month):
    if 5 <= month <= 10: return 'Rainy season (May–Oct)'
    else: return 'Dry season (Nov–Apr)'
df['Season'] = df['Month'].apply(define_season)
df = df.sort_values('Month')

# 3. Monthly mean PM2.5 (for trend line)
monthly_mean = df.groupby('Month')['pm2_5'].mean()

# Plot
fig = make_subplots(
    rows=2, cols=1,
    row_heights=[0.7, 0.3],
    vertical_spacing=0.12,
    subplot_titles=(
        "1. PM2.5 by month (with trend line)", 
        "2. Dry vs rainy season comparison"
    )
)

# Background season bands
# Rainy season (May–Oct)
fig.add_vrect(
    x0=4.5, x1=10.5,
    fillcolor="green", opacity=0.1, layer="below", line_width=0,
    annotation_text="RAINY SEASON (cleaner)", annotation_position="top left",
    row=1, col=1
)
# Dry season (two bands)
fig.add_vrect(
    x0=0.5, x1=4.5,
    fillcolor="red", opacity=0.05, layer="below", line_width=0,
    annotation_text="DRY SEASON", annotation_position="top left",
    row=1, col=1
)
fig.add_vrect(
    x0=10.5, x1=12.5,
    fillcolor="red", opacity=0.05, layer="below", line_width=0,
    row=1, col=1
)

# Boxplots by month
for m in range(1, 13):
    monthly_data = df[df['Month'] == m]
    color = '#2ca02c' if 5 <= m <= 10 else '#d62728' 
    
    fig.add_trace(
        go.Box(
            y=monthly_data['pm2_5'],
            x=[m] * len(monthly_data),
            name=f"Month {m}",
            marker=dict(color=color, size=3, opacity=0.5),
            boxpoints='outliers',
            line=dict(width=1.5),
            showlegend=False
        ),
        row=1, col=1
)

# Monthly trend line
fig.add_trace(
    go.Scatter(
        x=monthly_mean.index,
        y=monthly_mean.values,
        mode='lines+markers',
        name='Monthly mean',
        line=dict(color='black', width=3, dash='dot'),
        marker=dict(size=8, color='black'),
        hovertemplate="Mean month %{x}: %{y:.1f} µg/m³<extra></extra>"
    ),
    row=1, col=1
)

# Season comparison (horizontal boxplots)
# Dry season
fig.add_trace(
    go.Box(
        x=df[df['Season'] == 'Dry season (Nov–Apr)']['pm2_5'],
        name='Dry season',
        marker=dict(color='#d62728'),
        orientation='h',
        boxpoints=False
    ),
    row=2, col=1
)

# Rainy season
fig.add_trace(
    go.Box(
        x=df[df['Season'] == 'Rainy season (May–Oct)']['pm2_5'],
        name='Rainy season',
        marker=dict(color='#2ca02c'),
        orientation='h',
        boxpoints=False
    ),
    row=2, col=1
)

# Layout
fig.update_layout(
    title="<b>Seasonality analysis of PM2.5</b>",
    title_font=dict(size=22),
    height=900,
    template="plotly_white",
    showlegend=True,
    legend=dict(orientation="h", y=1.02, x=1, xanchor="right")
)

# Axes
fig.update_xaxes(
    title_text="Month", 
    tickmode='array', 
    tickvals=list(range(1, 13)),
    ticktext=[f"M{i}" for i in range(1, 13)],
    row=1, col=1
)
fig.update_yaxes(title_text="PM2.5 (µg/m³)", row=1, col=1)

# WHO guideline line
fig.add_hline(y=15, line_dash="dash", line_color="green", annotation_text="WHO", row=1, col=1)

In [26]:
fig.write_image("../figures/seasonality_pm25.pdf")

In [27]:
df['Visual_Size'] = df['prcp'].apply(lambda x: (x * 1.5) + 5)  # minimum size = 5

# Bubble chart
fig = px.scatter(
    df,
    x='tavg',
    y='pm2_5',
    size='Visual_Size',
    color='wspd',
    color_continuous_scale='Jet',
    hover_name='date',
    hover_data={
        'Visual_Size': False,
        'tavg': ':.1f',
        'pm2_5': ':.1f',
        'prcp': ':.1f',
        'wspd': ':.1f'
    },
    labels={
        'tavg': 'Average temperature (°C)',
        'pm2_5': 'PM2.5 (µg/m³)',
        'prcp': 'Rainfall (mm)',
        'wspd': 'Wind speed (km/h)'
    },
    title="<b>Multivariate view: temperature - PM2.5 - rainfall (size) - wind (color)</b>",
    opacity=0.7
)

fig.update_layout(
    height=800,
    template="plotly_white",
    font=dict(size=12)
)

# Reference threshold line
fig.add_hline(y=50, line_dash="dot", line_color="gray", annotation_text="Poor air (50)")

# Annotation: high-risk zone (calm, dry air)
fig.add_annotation(
    text="High-risk accumulation zone:<br>Calm wind (dark blue) + no rain (small bubbles)",
    xref="paper", yref="paper",
    x=0.05, y=0.95,
    showarrow=False,
    font=dict(color="red", size=11),
    align="left",
    bgcolor="rgba(255, 255, 255, 0.8)",
    bordercolor="red", borderwidth=1
)

# Annotation: cleaner zone (rain or strong wind)
fig.add_annotation(
    text="Cleaner zone:<br>Heavy rain (large bubbles)<br>or strong wind (yellow/red colors)",
    xref="paper", yref="paper",
    x=0.95, y=0.05,
    showarrow=False,
    font=dict(color="green", size=11),
    align="right",
    bgcolor="rgba(255, 255, 255, 0.8)",
    bordercolor="green", borderwidth=1
)

In [28]:
fig.write_image("../figures/bubble_temp_rain_wind.pdf")

In [29]:
# 1. Group by month
df['Month'] = df['date'].dt.month
# Monthly mean concentrations for three key pollutants
monthly_data = df.groupby('Month')[['pm2_5', 'nitrogen_dioxide', 'sulphur_dioxide']].mean()

# 2. Convert to percentage share within each month
monthly_data['Total'] = monthly_data.sum(axis=1)

monthly_data['pct_pm25'] = (monthly_data['pm2_5'] / monthly_data['Total']) * 100
monthly_data['pct_no2'] = (monthly_data['nitrogen_dioxide'] / monthly_data['Total']) * 100
monthly_data['pct_so2'] = (monthly_data['sulphur_dioxide'] / monthly_data['Total']) * 100

# Stacked bar chart
fig = go.Figure()

# PM2.5
fig.add_trace(go.Bar(
    x=monthly_data.index,
    y=monthly_data['pct_pm25'],
    name='PM2.5 (fine particles)',
    marker_color='#d62728',
    hovertemplate="Month %{x}<br>PM2.5: <b>%{y:.1f}%</b><br>(Actual: %{customdata:.1f})<extra></extra>",
    customdata=monthly_data['pm2_5']
))

# NO2
fig.add_trace(go.Bar(
    x=monthly_data.index,
    y=monthly_data['pct_no2'],
    name='NO2 (traffic)',
    marker_color='#1f77b4',
    hovertemplate="Month %{x}<br>NO2: <b>%{y:.1f}%</b><br>(Actual: %{customdata:.1f})<extra></extra>",
    customdata=monthly_data['nitrogen_dioxide']
))

# SO2
fig.add_trace(go.Bar(
    x=monthly_data.index,
    y=monthly_data['pct_so2'],
    name='SO2 (industry)',
    marker_color='#ff7f0e',
    hovertemplate="Month %{x}<br>SO2: <b>%{y:.1f}%</b><br>(Actual: %{customdata:.1f})<extra></extra>",
    customdata=monthly_data['sulphur_dioxide']
))

fig.update_layout(
    title="<b>Monthly composition of key pollutants (% share)</b>",
    title_font=dict(size=20),
    barmode='stack',
    template="plotly_white",
    height=600,
    legend=dict(orientation="h", y=1.02, x=0.5, xanchor="center"),
    xaxis=dict(
        title="Month",
        tickmode='linear',
        tickvals=list(range(1, 13))
    ),
    yaxis=dict(
        title="Share (%)",
        range=[0, 100],
        ticksuffix="%"
    )
)

# Highlight rainy season (May–Oct)
fig.add_shape(
    type="rect",
    x0=4.5, x1=10.5, y0=0, y1=100,
    fillcolor="green", opacity=0.05, layer="below", line_width=0,
)
fig.add_annotation(
    text="Rainy season (May–Oct)",
    x=7.5, y=105, showarrow=False,
    font=dict(color="green", size=10)
)

# Reference line at 50%
fig.add_hline(y=50, line_dash="dot", line_color="white", opacity=0.5)

In [30]:
fig.write_image("../figures/stacked_pollutants_pct.pdf")